# 编码器模块演示 (Encoders Module)

本notebook演示hscredit库中编码器模块的全部功能，包含8种编码方法。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n目标列分布:\n{df['FPD15'].value_counts()}")

/Users/xiaoxi/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 准备数据
X = df[feature_cols]
y = df[target_col]
print(f"X形状: {X.shape}, y形状: {y.shape}")

特征数量: 80
X形状: (12448, 80), y形状: (12448,)


## 1. 导入编码器模块

所有编码器均遵循sklearn Transformer接口规范。

In [3]:
from hscredit.core.encoders import (
    WOEEncoder,
    TargetEncoder,
    CountEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    QuantileEncoder,
    CatBoostEncoder,
    GBMEncoder
)
print("所有编码器已导入成功！")

所有编码器已导入成功！


## 2. WOE编码器 (Weight of Evidence)

WOE编码是风控建模中最常用的编码方式，将特征值转换为证据权重。

In [4]:
# WOE编码演示
demo_feature = "bj_qy24"
print(f"演示特征: {demo_feature}")


from hscredit.core import OptimalBinning
X[demo_feature]  = OptimalBinning(method="mdlp", max_n_bins=5).fit_transform(X[demo_feature], y, metric='indices')

encoder_woe = WOEEncoder()
X_woe = encoder_woe.fit_transform(X[[demo_feature]], y)
print(f"\nWOE编码后形状: {X_woe.shape}")
print(f"\nWOE编码结果（前10行）:")
print(X_woe.head(10))

演示特征: bj_qy24

WOE编码后形状: (12448, 1)

WOE编码结果（前10行）:
   bj_qy24
0   0.1846
1  -0.1806
2   0.1846
3  -0.1806
4  -0.1806
5  -0.4425
6  -0.1806
7   0.6043
8  -0.1806
9  -0.1806


In [5]:
# 查看WOE编码的映射表
print("WOE编码映射表:")
woe_mapping = encoder_woe.mapping_
print(woe_mapping)

WOE编码映射表:
{'bj_qy24': {3: 0.18459729219880552, 2: -0.18058061850055437, 0: -0.4424952319255252, 4: 0.6042743375282263, 1: -1.4478915823924245, nan: 0.0, '__UNKNOWN__': 0.0}}


## 3. 目标编码器 (Target Encoder)

基于目标变量均值进行编码，可添加平滑处理。

In [6]:
# 目标编码演示
encoder_target = TargetEncoder(smoothing=1.0)
X_target = encoder_target.fit_transform(X[[demo_feature]], y)
print(f"目标编码结果（前10行）:")
print(X_target.head(10))

print(f"\n目标编码映射:")
print(encoder_target.mapping_)

目标编码结果（前10行）:
   bj_qy24
0   0.0557
1   0.0785
2   0.0557
3   0.0785
4   0.0785
5   0.0982
6   0.0785
7   0.0369
8   0.0785
9   0.0785

目标编码映射:
{'bj_qy24': {0: 0.09823840762477568, 1: 0.22431007107213063, 2: 0.07846494363093069, 3: 0.0557310511520023, 4: 0.036888400802282555, nan: 0.0663560411311054, '__UNKNOWN__': 0.0663560411311054}}


## 4. 计数编码器 (Count Encoder)

用类别出现的次数进行编码，适用于高基数类别特征。

In [7]:
# 计数编码演示
encoder_count = CountEncoder()
X_count = encoder_count.fit_transform(X[[demo_feature]], y)
print(f"计数编码结果（前10行）:")
print(X_count.head(10))

print(f"\n计数编码映射:")
print(encoder_count.mapping_)

计数编码结果（前10行）:
   bj_qy24
0     4809
1     5480
2     4809
3     5480
4     5480
5      529
6     5480
7     1546
8     5480
9     5480

计数编码映射:
{'bj_qy24': {2: 5480, 3: 4809, 4: 1546, 0: 529, 1: 84, nan: 0, '__UNKNOWN__': 0}}


## 5. 独热编码器 (One-Hot Encoder)

将类别特征转换为独热向量。

In [8]:
# 独热编码演示
encoder_onehot = OneHotEncoder()
X_onehot = encoder_onehot.fit_transform(X[[demo_feature]])
print(f"独热编码后形状: {X_onehot.shape}")
print(f"\n独热编码结果（前5行）:")
print(X_onehot[:5])

print(encoder_onehot.mapping_)

独热编码后形状: (12448, 5)

独热编码结果（前5行）:
   bj_qy24_0  bj_qy24_1  bj_qy24_2  bj_qy24_3  bj_qy24_4
0          0          0          0          1          0
1          0          0          1          0          0
2          0          0          0          1          0
3          0          0          1          0          0
4          0          0          1          0          0
{'bj_qy24': {0: 'bj_qy24_0', 1: 'bj_qy24_1', 2: 'bj_qy24_2', 3: 'bj_qy24_3', 4: 'bj_qy24_4'}}


## 6. 序数编码器 (Ordinal Encoder)

将类别按顺序转换为整数。

In [9]:
# 序数编码演示
encoder_ordinal = OrdinalEncoder()
X_ordinal = encoder_ordinal.fit_transform(X[[demo_feature]])
print(f"序数编码结果（前10行）:")
print(X_ordinal[:10])

print(f"\n序数编码映射:")
print(encoder_ordinal.mapping_)

序数编码结果（前10行）:
   bj_qy24
0        3
1        2
2        3
3        2
4        2
5        0
6        2
7        4
8        2
9        2

序数编码映射:
{'bj_qy24': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, nan: -1, '__UNKNOWN__': -1}}


## 7. 分位数编码器 (Quantile Encoder)

基于分位数进行编码，适合连续变量分箱后的编码。

In [10]:
# 分位数编码演示
encoder_quantile = QuantileEncoder(quantile=0.2)
X_quantile = encoder_quantile.fit_transform(X[[demo_feature]], y)
print(f"分位数编码结果（前10行）:")
print(X_quantile.head(10))

encoder_quantile.mapping_

分位数编码结果（前10行）:
   bj_qy24
0   0.0000
1   0.0000
2   0.0000
3   0.0000
4   0.0000
5   0.0000
6   0.0000
7   0.0000
8   0.0000
9   0.0000


{'bj_qy24': {3: 0.0,
  2: 0.0,
  0: 0.0,
  4: 0.0,
  1: 0.0,
  nan: 0.0,
  '__UNKNOWN__': 0.0}}

## 8. CatBoost编码器

CatBoost风格的编码方式，使用有序目标统计。

In [11]:
# CatBoost编码演示
encoder_catboost = CatBoostEncoder()
X_catboost = encoder_catboost.fit_transform(X[[demo_feature]], y)
print(f"CatBoost编码结果（前10行）:")
print(X_catboost.head(10))

print(f"\nCatBoost编码映射:")
print(encoder_catboost.mapping_)

CatBoost编码结果（前10行）:
   bj_qy24
0   0.0532
1   0.0802
2   0.0549
3   0.0743
4   0.0783
5   0.0940
6   0.0811
7   0.0389
8   0.0561
9   0.0797

CatBoost编码映射:
{'bj_qy24': {0: 0.09829867674858223, 1: 0.2261904761904762, 2: 0.07846715328467153, 3: 0.05572884175504263, 4: 0.03686934023285899, nan: 0.0663560411311054, '__UNKNOWN__': 0.0663560411311054}}


## 9. GBM编码器

基于梯度提升树模型的编码，支持XGBoost/LightGBM/CatBoost。

In [12]:
# GBM编码演示
encoder_gbm = GBMEncoder(model_type='xgboost', n_estimators=100)
X_gbm = encoder_gbm.fit_transform(X[[demo_feature]], y)
print(f"GBM编码结果（前10行）:")
print(X_gbm.head(10))

print(encoder_gbm.mapping_)

GBM编码结果（前10行）:
   gbm_tree_0  gbm_tree_1  gbm_tree_2  gbm_tree_3  gbm_tree_4  gbm_tree_5  gbm_tree_6  gbm_tree_7  gbm_tree_8  gbm_tree_9  ...  gbm_tree_90  gbm_tree_91  gbm_tree_92  gbm_tree_93  gbm_tree_94  gbm_tree_95  gbm_tree_96  gbm_tree_97  gbm_tree_98  gbm_tree_99
0      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      6.0000  ...       6.0000       4.0000       6.0000       4.0000       4.0000       5.0000       6.0000       6.0000       6.0000       6.0000
1      4.0000      4.0000      4.0000      4.0000      4.0000      4.0000      4.0000      4.0000      4.0000      5.0000  ...       5.0000       3.0000       5.0000       3.0000       3.0000       4.0000       5.0000       5.0000       5.0000       5.0000
2      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      5.0000      6.0000  ...       6.0000       4.0000       6.0000       4.0000       4.0000       5.0000       6.0

## 10. 批量编码

对多个特征同时进行编码。

In [13]:
# 选择多个数值特征进行批量编码
numeric_features = feature_cols[:5]
print(f"选择特征: {numeric_features}")

# WOE批量编码
encoder_woe_batch = WOEEncoder()
X_woe_batch = encoder_woe_batch.fit_transform(X[numeric_features], y)
print(f"\n批量WOE编码后形状: {X_woe_batch.shape}")
print(f"\n批量WOE编码结果（前5行）:")
print(X_woe_batch.head())

print(encoder_woe_batch.mapping_)

选择特征: ['bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro']

批量WOE编码后形状: (12448, 5)

批量WOE编码结果（前5行）:
   bj_qy24  mobtech80  bairong_v1  xiaoniu_c3  dxm_v6pro
0   0.1846     0.3357     -1.9487      0.1308     0.0000
1  -0.1806     0.1148     -1.9487     -0.0028    -1.9487
2   0.1846     0.3357     -1.9487      1.0471     0.0000
3  -0.1806    -0.2887     -1.9487     -0.8501     0.0000
4  -0.1806     0.3357     -1.9487      0.5362     0.0000
{'bj_qy24': {3: 0.18459729219880552, 2: -0.18058061850055437, 0: -0.4424952319255252, 4: 0.6042743375282263, 1: -1.4478915823924245, nan: 0.0, '__UNKNOWN__': 0.0}, 'mobtech80': {718: 0.33573266179435224, 709: 0.11475724404326006, 698: -0.28872734368582703, 673: 0.06305649731997116, 701: -0.40673484023618667, 658: -0.3800509523910685, 714: -0.11719713540929419, 668: -0.5781208661531623, 696: -0.5830771825467144, 666: -0.2140658149168074, 676: -0.08329558373361287, -999: -0.7700118739632676, 692: -0.7959873603665284, 641: -1.137736654088585,

## 11. 编码器比较

对比不同编码器在相同特征上的效果。

In [14]:
# 编码器比较
encoders = {
    'WOE': WOEEncoder(),
    'Target': TargetEncoder(smoothing=1.0),
    'Count': CountEncoder(),
    'Quantile': QuantileEncoder(quantile=0.2),
    'CatBoost': CatBoostEncoder()
}

comparison_results = {}
for name, enc in encoders.items():
    X_encoded = enc.fit_transform(X[[demo_feature]], y)
    comparison_results[name] = X_encoded.values.flatten()[:10]

comparison_df = pd.DataFrame(comparison_results, index=range(10))
print("编码器效果对比（前10行）:")
comparison_df.round(4)

编码器效果对比（前10行）:


,WOE,Target,Count,Quantile,CatBoost
0,0.1846,0.0557,4809,0.0000,0.0524
1,-0.1806,0.0785,5480,0.0000,0.0787
2,0.1846,0.0557,4809,0.0000,0.0517
3,-0.1806,0.0785,5480,0.0000,0.0788
4,-0.1806,0.0785,5480,0.0000,0.0714
5,-0.4425,0.0982,529,0.0000,0.0835
6,-0.1806,0.0785,5480,0.0000,0.0799
7,0.6043,0.0369,1546,0.0000,0.0376
8,-0.1806,0.0785,5480,0.0000,0.0805
9,-0.1806,0.0785,5480,0.0000,0.0803


## 12. 保存编码结果

将编码后的特征保存到Excel文件。

In [16]:
# 保存编码结果
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/encoded_features.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    X_woe_batch.to_excel(writer, sheet_name='WOE_Encoded', index=False)
    comparison_df.to_excel(writer, sheet_name='Encoder_Comparison', index=False)

print(f"编码结果已保存至: {output_path}")

编码结果已保存至: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/encoded_features.xlsx
